<a href="https://colab.research.google.com/github/fwitschel/HCIBM/blob/main/notebooks/Explainable_start_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Execute this code only if in colab
if 'COLAB_GPU' in os.environ:
  print("Executing in Colab!")
  # Cloning GitHub repository
  !git clone https://github.com/fwitschel/HCIBM.git
  %cd HCIBM


# Library imports

In [ ]:
!pip install lime
!pip install imodels

import shap
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import lime
import lime.lime_tabular
from xgboost import XGBClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn import tree
from imodels import RuleFitRegressor
from imodels import RuleFitClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

import warnings
# suppresses deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Data loading and cleaning

The first step is loading the dataset

In [ ]:
# load the data
data = pd.read_csv('/content/HCIBM/datasets/Myocardial infarction complications Database.csv')

These columns are removed as they refer to other complications that in this teaching case won't be used.

In [ ]:
# remove the columns corresponding to other complications that we do not want to predict
data = data.drop('KFK_BLOOD', axis=1)
data = data.drop('IBS_NASL', axis=1)
data = data.drop('LET_IS', axis=1)
data = data.drop('P_IM_STEN', axis=1)
data = data.drop('REC_IM', axis=1)
data = data.drop('DRESSLER', axis=1)
data = data.drop('RAZRIV', axis=1)
data = data.drop('OTEK_LANC', axis=1)
data = data.drop('A_V_BLOK', axis=1)
data = data.drop('FIBR_JELUD', axis=1)
data = data.drop('JELUD_TAH', axis=1)
data = data.drop('PREDS_TAH', axis=1)
data = data.drop('FIBR_PREDS', axis=1)
data = data.drop('ID', axis=1)

# we additionally drop some columns where a lot of values are missing
# (makes performance of models better...)
data = data.drop('S_AD_KBRIG', axis=1)
data = data.drop('D_AD_KBRIG', axis=1)
data = data.drop('NOT_NA_KB', axis=1)
data = data.drop('LID_KB', axis=1)
data = data.drop('NA_KB', axis=1)
data = data.drop('GIPO_K', axis=1)

In [ ]:
# set target and one-hot encode categorical stuff:
X = data.drop('ZSN', axis=1)
one_hot_X = pd.get_dummies(X)
y = data.ZSN

In [ ]:
# train-test split and fitting an XGBoost model
X_train, X_test, y_train, y_test = train_test_split(one_hot_X, y, test_size=0.2)
model = XGBClassifier()
model.fit(X_train.values, y_train.values)